This notebook is for creation of plots that comapre cooled distribtuions to repex data (eg 4a), plot cooled and equilbirum (277K and 230K) histograms for a given number of states (eg 4b), and plot a  event time heatmap (eg 4c)\

Below this cell is also one for producing a full histogram and full event time heatmap.

This notebook can be used to produce s9 and s18 

In [ ]:
from __future__ import annotations

import re
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import ticker
from matplotlib.colors import Normalize
from matplotlib.patches import Patch
from matplotlib.ticker import AutoMinorLocator


# =============================================================================
# USER SETTINGS
# =============================================================================

input_root = Path("./input").resolve()

processing_output_directory = Path(
    "./output/cooling-data-processing"
).resolve()

figure_output_directory = Path(
    "./output/cooling-analysis"
).resolve()

k_choice = 50
tica_lag = 250
chosen_lag = 400

population_threshold = 0.01
event_time_divisor = 20.0

cooling_order = ["6e9", "2e9", "6e8", "2e8", "6e7", "2e7", "6e6"]
panel_a_labels = ["6e9", "6e8", "6e7", "6e6"]

cooling_pretty_labels = {
    "6e9": "6.4 e9 K/s",
    "2e9": "2.7 e9 K/s",
    "6e8": "6.4 e8 K/s",
    "2e8": "2.7 e8 K/s",
    "6e7": "6.4 e7 K/s",
    "2e7": "2.7 e7 K/s",
    "6e6": "6.4 e6 K/s",
}

cooling_time_to_230_labels = {
    "6e9": "7.3 ns",
    "2e9": "18 ns",
    "6e8": "73 ns",
    "2e8": "179 ns",
    "6e7": "734 ns",
    "2e7": "1730 ns",
    "6e6": "7340 ns",
}

vit_colors = {
    "6e9": "#fde725",
    "2e9": "#90d743",
    "6e8": "#35b779",
    "2e8": "#21918c",
    "6e7": "#31688e",
    "2e7": "#443983",
    "6e6": "#440154",
    "277K": "tab:red",
    "230K": "tab:blue",
}


# =============================================================================
# INPUTS PRODUCED BY cooling-data-processing.ipynb
# =============================================================================

figure_output_directory.mkdir(
    parents=True,
    exist_ok=True,
)

cooling_csv = processing_output_directory / (
    f"cooling_pairs_summary_k{k_choice}_t{tica_lag}_lag_{chosen_lag}.csv"
)

bootstrap_npz = processing_output_directory / (
    f"cooling_bootstraps_k{k_choice}_t{tica_lag}_lag_{chosen_lag}.npz"
)

event_time_csv = (
    processing_output_directory
    / "event_time_per_state_da.csv"
)

# Panel A still needs the equilibrium-temperature dtrajs.
state_model_directory = (
    input_root
    / "state-model"
    / f"t{tica_lag}_k{k_choice}"
)

repex_directory = (
    state_model_directory
    / "repex-dtrajs"
)

figure_path = figure_output_directory / (
    f"si_figure_kl_histogram_binned_flux_reference_style_"
    f"k{k_choice}_t{tica_lag}.pdf"
)

for path in [cooling_csv, bootstrap_npz, event_time_csv]:
    if not path.is_file():
        raise FileNotFoundError(
            f"Required output from the previous cell is missing: {path}"
        )

if not repex_directory.is_dir():
    raise FileNotFoundError(
        f"Replica-exchange dtraj directory not found: {repex_directory}"
    )


# =============================================================================
# PLOT SETTINGS
# =============================================================================

kl_epsilon = 1.0e-12
kl_ymin = 1.0e-2
kl_ymax_pad_fraction = 0.05
bootstrap_percentiles = [2.5, 97.5]

heatmap_epsilon = 1.0e-12
heatmap_cmap = "PRGn"
heatmap_vmax_percentile = 95.0
progressive_base_width_ns = 1000.0
progressive_group_size = 3

annotate_heatmap = True
annotate_with_ci = False
annotate_fontsize = 5.0
annotate_decimals = 1
show_bin_counts = False

axis_label_fontsize = 9
tick_labelsize = 7
panel_label_fontsize = 13
panel_c_heading_fontsize = 9
legend_fontsize = 6

err_capsize = 1
err_kw = {
    "elinewidth": 1,
    "capthick": 1,
    "alpha": 0.95,
}


# =============================================================================
# HELPERS
# =============================================================================

def format_axes(
    ax,
    label_size=9,
    tick_size=7,
    stroke_in=0.01,
):
    width = stroke_in * 72.0

    ax.minorticks_on()

    ax.xaxis.set_minor_locator(
        ticker.LogLocator(subs="auto")
        if ax.get_xscale() == "log"
        else AutoMinorLocator(2)
    )

    ax.yaxis.set_minor_locator(
        ticker.LogLocator(subs="auto")
        if ax.get_yscale() == "log"
        else AutoMinorLocator(2)
    )

    for spine in ax.spines.values():
        spine.set_linewidth(1.5)
        spine.set_color("black")

    ax.tick_params(
        which="major",
        width=width,
        length=4.5,
        direction="in",
        top=True,
        bottom=True,
        left=True,
        right=True,
    )

    ax.tick_params(
        which="minor",
        width=width * 0.75,
        length=2.25,
        direction="in",
        top=True,
        bottom=True,
        left=True,
        right=True,
    )

    ax.xaxis.label.set_size(label_size)
    ax.yaxis.label.set_size(label_size)

    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontsize(tick_size)


def extract_population_series(
    dataframe,
    label,
):
    subset = dataframe.loc[
        dataframe["lab"].str.lower() == label.lower()
    ].sort_values("state")

    expected_states = np.arange(k_choice)
    states = subset["state"].to_numpy(dtype=int)

    if subset.empty or not np.array_equal(
        states,
        expected_states,
    ):
        raise ValueError(
            f"Rows for {label!r} do not cover "
            f"states 0..{k_choice - 1}."
        )

    mean = subset["mean"].to_numpy(dtype=float)

    confidence_interval = np.vstack(
        (
            subset["ci_lower_95"].to_numpy(dtype=float),
            subset["ci_upper_95"].to_numpy(dtype=float),
        )
    )

    return mean, confidence_interval


def normalized_probability(values):
    values = np.clip(
        np.asarray(values, dtype=float),
        0.0,
        np.inf,
    )

    total = float(values.sum())

    if not np.isfinite(total) or total <= 0.0:
        raise ValueError(
            "Probability vector has non-positive total mass."
        )

    return values / total


def kl_to_reference(
    population,
    reference,
):
    population = normalized_probability(population)
    reference = normalized_probability(reference)

    population = np.clip(
        population,
        kl_epsilon,
        np.inf,
    )

    reference = np.clip(
        reference,
        kl_epsilon,
        np.inf,
    )

    population /= population.sum()
    reference /= reference.sum()

    return float(
        np.sum(
            population
            * np.log(
                population / reference
            )
        )
    )


def kl_batch_to_reference(
    samples,
    reference,
):
    samples = np.clip(
        np.asarray(samples, dtype=float),
        0.0,
        np.inf,
    )

    row_sums = samples.sum(
        axis=1,
        keepdims=True,
    )

    if (
        np.any(row_sums <= 0.0)
        or np.any(~np.isfinite(row_sums))
    ):
        raise ValueError(
            "Bootstrap population array contains an invalid row."
        )

    samples /= row_sums

    reference = normalized_probability(reference)

    samples = np.clip(
        samples,
        kl_epsilon,
        np.inf,
    )

    reference = np.clip(
        reference,
        kl_epsilon,
        np.inf,
    )

    samples /= samples.sum(
        axis=1,
        keepdims=True,
    )

    reference /= reference.sum()

    return np.sum(
        samples
        * np.log(
            samples / reference[None, :]
        ),
        axis=1,
    )


def load_dtraj_population(path):
    with h5py.File(path, "r") as handle:
        dataset = None

        for key in [
            "arr_00",
            "dtraj",
            "dtrajs",
            "traj",
            "assignments",
            "0",
        ]:
            if key in handle:
                dataset = handle[key][:]
                break

        if dataset is None:
            for value in handle.values():
                if (
                    isinstance(value, h5py.Dataset)
                    and np.issubdtype(
                        value.dtype,
                        np.integer,
                    )
                ):
                    dataset = value[:]
                    break

    if dataset is None:
        raise ValueError(
            f"No integer dtraj dataset found in {path}."
        )

    states = np.asarray(
        dataset
    ).reshape(
        -1
    ).astype(
        int
    )

    states = states[
        (states >= 0)
        & (states < k_choice)
    ]

    counts = np.bincount(
        states,
        minlength=k_choice,
    ).astype(
        float
    )

    if counts.sum() <= 0.0:
        raise ValueError(
            f"No valid state assignments found in {path}."
        )

    return counts / counts.sum()


def discover_ladder_files():
    pattern = re.compile(
        r"^(\d+)k-dtraj\.h5$"
    )

    found = {}

    for path in repex_directory.iterdir():
        match = pattern.match(path.name)

        if path.is_file() and match:
            temperature = int(
                match.group(1)
            )

            if 230 <= temperature <= 277:
                found[temperature] = path

    if not found:
        raise FileNotFoundError(
            f"No 230–277 K dtraj files found in "
            f"{repex_directory}."
        )

    return dict(
        sorted(
            found.items()
        )
    )


def yerr(
    mean,
    confidence_interval,
):
    return np.vstack(
        (
            np.maximum(
                mean - confidence_interval[0],
                0.0,
            ),
            np.maximum(
                confidence_interval[1] - mean,
                0.0,
            ),
        )
    )


def progressive_edges(
    maximum,
    base_width,
    group_size,
):
    edges = [0.0]
    width = float(base_width)
    count = 0

    while edges[-1] < maximum:
        edges.append(
            edges[-1] + width
        )

        count += 1

        if count == group_size:
            width *= 2.0
            count = 0

    return np.asarray(
        edges,
        dtype=float,
    )


def text_colour(
    value,
    norm,
):
    red, green, blue, _ = plt.get_cmap(
        heatmap_cmap
    )(
        norm(
            float(value)
        )
    )

    luminance = (
        0.2126 * red
        + 0.7152 * green
        + 0.0722 * blue
    )

    return (
        "white"
        if luminance < 0.5
        else "black"
    )


# =============================================================================
# LOAD THE THREE OUTPUTS FROM THE PREVIOUS CELL
# =============================================================================

cooling_df = pd.read_csv(
    cooling_csv
)

required_columns = {
    "state",
    "lab",
    "mean",
    "ci_lower_95",
    "ci_upper_95",
}

missing_columns = (
    required_columns
    - set(cooling_df.columns)
)

if missing_columns:
    raise ValueError(
        f"Cooling CSV is missing columns: "
        f"{sorted(missing_columns)}"
    )

cooling_df["state"] = cooling_df[
    "state"
].astype(
    int
)

cooling_df["lab"] = cooling_df[
    "lab"
].astype(
    str
).str.strip()

if not np.array_equal(
    np.sort(
        cooling_df["state"].unique()
    ),
    np.arange(k_choice),
):
    raise ValueError(
        f"Cooling CSV does not contain "
        f"states 0..{k_choice - 1}."
    )

event_df = pd.read_csv(
    event_time_csv
).sort_values(
    "state"
).reset_index(
    drop=True
)

event_columns = {
    "state",
    "event_time_median_da_ns",
    "event_time_q025_da_ns",
    "event_time_q975_da_ns",
}

missing_columns = (
    event_columns
    - set(event_df.columns)
)

if missing_columns:
    raise ValueError(
        f"Event-time CSV is missing columns: "
        f"{sorted(missing_columns)}"
    )

if not np.array_equal(
    event_df["state"].to_numpy(dtype=int),
    np.arange(k_choice),
):
    raise ValueError(
        f"Event-time CSV does not contain "
        f"states 0..{k_choice - 1}."
    )

pi_277, pi_277_ci = extract_population_series(
    cooling_df,
    "pi_277K",
)

pi_230, pi_230_ci = extract_population_series(
    cooling_df,
    "pi_230K",
)

cooled_mean = {}
cooled_ci = {}

for label in cooling_order:
    (
        cooled_mean[label],
        cooled_ci[label],
    ) = extract_population_series(
        cooling_df,
        label,
    )

with np.load(
    bootstrap_npz,
    allow_pickle=True,
) as archive:
    cooled_bootstrap = {}

    for label in panel_a_labels:
        key = f"cooled_{label}"

        if key not in archive.files:
            raise KeyError(
                f"{bootstrap_npz} is missing {key!r}."
            )

        samples = np.asarray(
            archive[key],
            dtype=float,
        )

        if (
            samples.ndim != 2
            or samples.shape[1] != k_choice
        ):
            raise ValueError(
                f"{key!r} has shape {samples.shape}; "
                f"expected (n_boot, {k_choice})."
            )

        cooled_bootstrap[label] = samples


# =============================================================================
# PANEL A DATA
# =============================================================================

ladder_files = discover_ladder_files()
temperatures = list(ladder_files)

ladder_populations = {
    temperature: load_dtraj_population(path)
    for temperature, path in ladder_files.items()
}

metric_mean = np.empty(
    (
        len(panel_a_labels),
        len(temperatures),
    ),
    dtype=float,
)

band_low = np.empty_like(
    metric_mean
)

band_high = np.empty_like(
    metric_mean
)

for row, label in enumerate(
    panel_a_labels
):
    for column, temperature in enumerate(
        temperatures
    ):
        reference = ladder_populations[
            temperature
        ]

        metric_mean[
            row,
            column,
        ] = kl_to_reference(
            cooled_mean[label],
            reference,
        )

        metric_samples = kl_batch_to_reference(
            cooled_bootstrap[label],
            reference,
        )

        (
            band_low[row, column],
            band_high[row, column],
        ) = np.percentile(
            metric_samples,
            bootstrap_percentiles,
        )

kl_ymax = float(
    np.nanmax(
        band_high
    )
) * (
    1.0
    + kl_ymax_pad_fraction
)

if (
    not np.isfinite(kl_ymax)
    or kl_ymax <= kl_ymin
):
    raise ValueError(
        "Invalid KL y-axis range."
    )


# =============================================================================
# PANEL B DATA
# =============================================================================

selected_states = np.where(
    pi_277 > population_threshold
)[0]

selected_states = selected_states[
    np.argsort(
        pi_277[selected_states]
    )[::-1]
]

if selected_states.size == 0:
    raise ValueError(
        f"No states have pi_277K > "
        f"{population_threshold:.2%}."
    )

all_upper_values = [
    pi_277_ci[
        1,
        selected_states,
    ],
    pi_230_ci[
        1,
        selected_states,
    ],
]

all_upper_values.extend(
    cooled_ci[label][
        1,
        selected_states,
    ]
    for label in cooling_order
)

histogram_ymax = max(
    float(
        np.max(values)
    )
    for values in all_upper_values
) * 1.06

histogram_keys = (
    ["277K"]
    + cooling_order
    + ["230K"]
)

histogram_labels = (
    ["Equilibrium 277 K"]
    + [
        cooling_pretty_labels[label]
        for label in cooling_order
    ]
    + ["Equilibrium 230 K"]
)

histogram_colours = [
    vit_colors[key]
    for key in histogram_keys
]

histogram_width = (
    0.8
    / len(histogram_keys)
)

histogram_offsets = (
    np.arange(
        len(histogram_keys)
    )
    - (
        len(histogram_keys) - 1
    )
    / 2.0
) * histogram_width


# =============================================================================
# PANEL C DATA
# =============================================================================

base = pd.Series(
    pi_277,
    index=np.arange(k_choice),
    name="baseline_mean",
)

working = cooling_df.loc[
    ~cooling_df["lab"]
    .str.lower()
    .isin(
        {
            "pi_277k",
            "pi_230k",
        }
    )
].merge(
    base.rename_axis(
        "state"
    ).reset_index(),
    on="state",
    how="left",
)

p = np.maximum(
    working[
        "baseline_mean"
    ].to_numpy(float),
    heatmap_epsilon,
)

q = np.maximum(
    working[
        "mean"
    ].to_numpy(float),
    heatmap_epsilon,
)

q_low = np.maximum(
    working[
        "ci_lower_95"
    ].to_numpy(float),
    heatmap_epsilon,
)

q_high = np.maximum(
    working[
        "ci_upper_95"
    ].to_numpy(float),
    heatmap_epsilon,
)

working["center"] = (
    q
    * np.log(
        q / p
    )
)

working["lower"] = (
    q_low
    * np.log(
        q_low / p
    )
)

working["upper"] = (
    q_high
    * np.log(
        q_high / p
    )
)

column_map = {
    label.lower(): label
    for label in working["lab"].unique()
}

ordered_columns = [
    column_map[
        label.lower()
    ]
    for label in cooling_order
]


def pivot_value(column):
    return (
        working.pivot(
            index="state",
            columns="lab",
            values=column,
        )
        .reindex(
            index=np.arange(k_choice),
            columns=ordered_columns,
        )
    )


state_center = pivot_value(
    "center"
)

state_lower = pivot_value(
    "lower"
)

state_upper = pivot_value(
    "upper"
)

event_time_us = (
    event_df[
        "event_time_median_da_ns"
    ].to_numpy(
        dtype=float
    )
    / event_time_divisor
    / 1000.0
)

base_width_us = (
    progressive_base_width_ns
    / event_time_divisor
    / 1000.0
)

bin_edges = progressive_edges(
    float(
        np.nanmax(
            event_time_us
        )
    ),
    base_width_us,
    progressive_group_size,
)

bin_index = np.clip(
    np.searchsorted(
        bin_edges,
        event_time_us,
        side="right",
    )
    - 1,
    0,
    len(bin_edges) - 2,
)

used_bins = sorted(
    np.unique(
        bin_index
    ).astype(
        int
    ).tolist()
)

center_rows = []
lower_rows = []
upper_rows = []
bin_counts = {}

for bin_number in used_bins:
    states = np.where(
        bin_index == bin_number
    )[0]

    bin_counts[
        bin_number
    ] = int(
        states.size
    )

    center_rows.append(
        np.nanmedian(
            state_center.iloc[
                states
            ].to_numpy(
                float
            ),
            axis=0,
        )
    )

    lower_rows.append(
        np.nanmedian(
            state_lower.iloc[
                states
            ].to_numpy(
                float
            ),
            axis=0,
        )
    )

    upper_rows.append(
        np.nanmedian(
            state_upper.iloc[
                states
            ].to_numpy(
                float
            ),
            axis=0,
        )
    )

heatmap_center = pd.DataFrame(
    center_rows,
    index=used_bins,
    columns=ordered_columns,
)

heatmap_lower = pd.DataFrame(
    lower_rows,
    index=used_bins,
    columns=ordered_columns,
)

heatmap_upper = pd.DataFrame(
    upper_rows,
    index=used_bins,
    columns=ordered_columns,
)

heatmap_values = heatmap_center.to_numpy(
    dtype=float
)

heatmap_limit = float(
    np.nanpercentile(
        np.abs(
            heatmap_values
        ),
        heatmap_vmax_percentile,
    )
)

if (
    not np.isfinite(heatmap_limit)
    or heatmap_limit <= 0.0
):
    heatmap_limit = 1.0

heatmap_norm = Normalize(
    vmin=-heatmap_limit,
    vmax=heatmap_limit,
    clip=True,
)

heatmap_xlabels = [
    (
        f"{cooling_pretty_labels[label]}\n"
        f"({cooling_time_to_230_labels[label]})"
    )
    for label in cooling_order
]

heatmap_ylabels = []

for bin_number in used_bins:
    label = (
        f"{bin_edges[bin_number]:.2f}"
        f"–"
        f"{bin_edges[bin_number + 1]:.2f} µs"
    )

    if show_bin_counts:
        label += (
            f"\n[{bin_counts[bin_number]}]"
        )

    heatmap_ylabels.append(
        label
    )


# =============================================================================
# FIGURE GEOMETRY
# =============================================================================

panel_a_width = 3.2
panel_a_height = 1.6
panel_a_left = 0.60
panel_a_right = 0.25
panel_a_bottom = 0.55
panel_a_top_margin = 0.30

panel_b_width = panel_a_width
panel_b_height = 2.0 * panel_a_height
panel_b_left = panel_a_left
panel_b_bottom = panel_a_bottom
panel_b_top_margin = panel_a_top_margin

panel_c_width = 3.25
panel_c_left_margin = 1.20
panel_c_right_margin = 1.10
panel_gap = 0.01

panel_b_x = panel_b_left
panel_b_y = panel_b_bottom

panel_a_x = panel_a_left
panel_a_y = (
    panel_b_y
    + panel_b_height
    + panel_b_top_margin
    + panel_a_bottom
)

panel_c_x = (
    panel_a_left
    + panel_a_width
    + panel_a_right
    + panel_gap
    + panel_c_left_margin
)

panel_c_y = panel_b_y

panel_c_height = (
    panel_a_y
    + panel_a_height
    - panel_b_y
)

colorbar_pad = (
    0.025
    * panel_c_width
)

colorbar_width = (
    0.045
    * panel_c_width
)

colorbar_height = (
    0.45
    * panel_c_height
)

colorbar_x = (
    panel_c_x
    + panel_c_width
    + colorbar_pad
)

colorbar_y = (
    panel_c_y
    + 0.5
    * (
        panel_c_height
        - colorbar_height
    )
)

figure_width = (
    colorbar_x
    + colorbar_width
    + panel_c_right_margin
)

figure_height = (
    panel_a_y
    + panel_a_height
    + panel_a_top_margin
)

fig = plt.figure(
    figsize=(
        figure_width,
        figure_height,
    )
)

ax_a = fig.add_axes(
    [
        panel_a_x / figure_width,
        panel_a_y / figure_height,
        panel_a_width / figure_width,
        panel_a_height / figure_height,
    ]
)

ax_b = fig.add_axes(
    [
        panel_b_x / figure_width,
        panel_b_y / figure_height,
        panel_b_width / figure_width,
        panel_b_height / figure_height,
    ]
)

ax_c = fig.add_axes(
    [
        panel_c_x / figure_width,
        panel_c_y / figure_height,
        panel_c_width / figure_width,
        panel_c_height / figure_height,
    ]
)

colorbar_ax = fig.add_axes(
    [
        colorbar_x / figure_width,
        colorbar_y / figure_height,
        colorbar_width / figure_width,
        colorbar_height / figure_height,
    ]
)


# =============================================================================
# PANEL A
# =============================================================================

panel_a_handles = []

for row, label in enumerate(
    panel_a_labels
):
    values = np.maximum(
        metric_mean[row],
        kl_ymin,
    )

    lower = np.maximum(
        band_low[row],
        kl_ymin,
    )

    upper = np.maximum(
        band_high[row],
        kl_ymin,
    )

    line, = ax_a.plot(
        temperatures,
        values,
        marker="o",
        markersize=4.0,
        linewidth=1.1,
        color=vit_colors[label],
        label=cooling_pretty_labels[label],
    )

    ax_a.fill_between(
        temperatures,
        lower,
        upper,
        color=vit_colors[label],
        alpha=0.30,
        linewidth=0.0,
    )

    panel_a_handles.append(
        line
    )

ax_a.set_xlim(
    float(
        min(temperatures)
    ),
    float(
        max(temperatures)
    ),
)

ax_a.set_yscale(
    "log"
)

ax_a.set_ylim(
    kl_ymin,
    kl_ymax,
)

ax_a.set_xlabel(
    "Equilibrium Temperature (K)"
)

ax_a.set_ylabel(
    "KL Divergence to eq."
)

format_axes(
    ax_a
)

ax_a.legend(
    handles=panel_a_handles,
    loc="upper right",
    frameon=False,
    fontsize=legend_fontsize,
    handlelength=1.3,
    handletextpad=0.35,
    borderpad=0.15,
)


# =============================================================================
# PANEL B
# =============================================================================

positions = np.arange(
    selected_states.size,
    dtype=float,
)

series = [
    (
        pi_277,
        pi_277_ci,
    )
]

series.extend(
    (
        cooled_mean[label],
        cooled_ci[label],
    )
    for label in cooling_order
)

series.append(
    (
        pi_230,
        pi_230_ci,
    )
)

for index, (
    mean,
    confidence_interval,
) in enumerate(
    series
):
    selected_mean = mean[
        selected_states
    ]

    selected_ci = confidence_interval[
        :,
        selected_states,
    ]

    ax_b.bar(
        positions
        + histogram_offsets[index],
        selected_mean,
        histogram_width,
        color=histogram_colours[index],
        yerr=yerr(
            selected_mean,
            selected_ci,
        ),
        capsize=err_capsize,
        error_kw=err_kw,
    )

ax_b.set_xlim(
    -0.5,
    selected_states.size - 0.5,
)

ax_b.set_ylim(
    0.0,
    histogram_ymax,
)

ax_b.set_xticks(
    positions
)

ax_b.set_xticklabels(
    selected_states,
    rotation=45,
    ha="right",
)

ax_b.set_xlabel(
    "State Index"
)

ax_b.set_ylabel(
    "Probability"
)

format_axes(
    ax_b
)

ax_b.legend(
    handles=[
        Patch(
            facecolor=histogram_colours[index],
            edgecolor="none",
            label=label,
        )
        for index, label in enumerate(
            histogram_labels
        )
    ],
    loc="upper right",
    frameon=False,
    fontsize=legend_fontsize,
    handlelength=0.9,
    handletextpad=0.35,
    columnspacing=0.6,
    borderpad=0.15,
)


# =============================================================================
# PANEL C
# =============================================================================

image = ax_c.imshow(
    heatmap_values,
    aspect="auto",
    interpolation="nearest",
    cmap=heatmap_cmap,
    norm=heatmap_norm,
)

ax_c.set_xticks(
    np.arange(
        len(cooling_order)
    )
)

ax_c.set_xticklabels(
    heatmap_xlabels,
    rotation=45,
    ha="right",
)

ax_c.set_yticks(
    np.arange(
        len(used_bins)
    )
)

ax_c.set_yticklabels(
    heatmap_ylabels
)

ax_c.set_xlabel(
    "Cooling Rate\n(Time to 230 K)"
)

ax_c.set_ylabel(
    ""
)

format_axes(
    ax_c
)

ax_c.tick_params(
    axis="both",
    which="both",
    length=0,
)

if annotate_heatmap:
    number_format = (
        f"{{:.{annotate_decimals}e}}"
    )

    for row in range(
        heatmap_center.shape[0]
    ):
        for column in range(
            heatmap_center.shape[1]
        ):
            value = float(
                heatmap_center.iloc[
                    row,
                    column,
                ]
            )

            if not np.isfinite(value):
                continue

            if annotate_with_ci:
                label = (
                    f"{number_format.format(value)}\n"
                    f"["
                    f"{number_format.format(float(heatmap_lower.iloc[row, column]))},"
                    f"{number_format.format(float(heatmap_upper.iloc[row, column]))}"
                    f"]"
                )
            else:
                label = number_format.format(
                    value
                )

            ax_c.text(
                column,
                row,
                label,
                ha="center",
                va="center",
                fontsize=annotate_fontsize,
                color=text_colour(
                    value,
                    heatmap_norm,
                ),
            )

colorbar = fig.colorbar(
    image,
    cax=colorbar_ax,
)

colorbar.ax.tick_params(
    labelsize=tick_labelsize
)

colorbar.ax.yaxis.set_major_formatter(
    ticker.FuncFormatter(
        lambda value, position: (
            f"{value:.1e}"
        )
    )
)

colorbar.set_label(
    "Contribution to KL divergence relative to 277 K eq.",
    fontsize=axis_label_fontsize,
    labelpad=2,
)

colorbar.outline.set_linewidth(
    1.5
)


# =============================================================================
# PANEL LABELS AND SAVE
# =============================================================================

panel_a_top = (
    panel_a_y
    + panel_a_height
)

panel_b_top = (
    panel_b_y
    + panel_b_height
)

label_y_ac = (
    panel_a_top
    + 0.03
)

label_y_b = (
    panel_b_top
    + 0.03
)

fig.text(
    0.08 / figure_width,
    label_y_ac / figure_height,
    "A",
    fontsize=panel_label_fontsize,
    va="bottom",
    ha="left",
)

fig.text(
    0.08 / figure_width,
    label_y_b / figure_height,
    "B",
    fontsize=panel_label_fontsize,
    va="bottom",
    ha="left",
)

fig.text(
    (
        panel_c_x
        - panel_c_left_margin
        - 0.08
    )
    / figure_width,
    label_y_ac / figure_height,
    "C",
    fontsize=panel_label_fontsize,
    va="bottom",
    ha="left",
)

fig.text(
    (
        panel_c_x
        - 0.05
    )
    / figure_width,
    label_y_ac / figure_height,
    "Event Time\n(bin range)",
    fontsize=panel_c_heading_fontsize,
    va="bottom",
    ha="right",
    multialignment="center",
)

fig.savefig(
    figure_path,
    format="pdf",
)

print(
    f"Panel A temperatures: "
    f"{temperatures}"
)

print(
    f"Panel B selected "
    f"{selected_states.size} states with "
    f"pi_277K > {population_threshold:.2%}: "
    f"{selected_states.tolist()}"
)

print(
    f"Panel C used "
    f"{len(used_bins)} progressive "
    f"DA event-time bins."
)

print(
    f"Saved figure to: "
    f"{figure_path}"
)

plt.show()

This is for a full plot of histograms and event time not binned

In [ ]:
from __future__ import annotations

import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import ticker
from matplotlib.colors import Normalize
from matplotlib.patches import Patch
from matplotlib.ticker import AutoMinorLocator


# ============================================================
# USER SETTINGS
# ============================================================

# These are the local outputs produced by cooling-data-processing.ipynb.
processing_output_directory = Path(
    "./output/cooling-data-processing"
).resolve()

figure_output_directory = Path(
    "./output/cooling-analysis"
).resolve()

k_choice = 50
tica_lag = 250
chosen_lag = 400

event_time_csv = (
    processing_output_directory
    / "event_time_per_state_da.csv"
)

cooling_csv = (
    processing_output_directory
    / (
        f"cooling_pairs_summary_"
        f"k{k_choice}_"
        f"t{tica_lag}_"
        f"lag_{chosen_lag}.csv"
    )
)

figure_path = (
    figure_output_directory
    / (
        f"histograms_and_heatmaps_"
        f"k{k_choice}_"
        f"t{tica_lag}_"
        f"lag_{chosen_lag}.pdf"
    )
)

figure_output_directory.mkdir(
    parents=True,
    exist_ok=True,
)


# --------------------------
# Heatmap settings
# --------------------------

change_metric = "kl_qp_contrib"
baseline_label = "pi_277k"
exclude_labs = ["pi_230k"]

cooling_order = [
    "6e9",
    "2e9",
    "6e8",
    "2e8",
    "6e7",
    "2e7",
    "6e6",
]

cooling_pretty_labels = {
    "6e9": "6.4 e9 K/s",
    "2e9": "2.7 e9 K/s",
    "6e8": "6.4 e8 K/s",
    "2e8": "2.7 e8 K/s",
    "6e7": "6.4 e7 K/s",
    "2e7": "2.7 e7 K/s",
    "6e6": "6.4 e6 K/s",
}

scale_mode = "symmetric"
cmap = "PRGn"
vmax_percentile = 95.0

annotate = False
annotate_with_ci = True
annotate_fontsize = 4.0
annotate_decimals = 2

event_time_divisor = 20.0
event_time_label_decimals = 0


# --------------------------
# Left-panel data settings
# --------------------------

full_nrows = 5
full_ymax_pad_frac = 0.06

vit_colors = {
    "6e9": "#fde725",
    "2e9": "#90d743",
    "6e8": "#35b779",
    "2e8": "#21918c",
    "6e7": "#31688e",
    "2e7": "#443983",
    "6e6": "#440154",
    "277K": "tab:red",
    "230K": "tab:blue",
}

legend_enabled = True
legend_ncol = 3
legend_handlelength = 0.9
legend_handletextpad = 0.30
legend_columnspacing = 0.6
legend_borderpad = 0.15

err_capsize = 1

err_kw = {
    "elinewidth": 1,
    "capthick": 1,
    "alpha": 0.95,
}


# --------------------------
# Figure/layout settings
# --------------------------

figure_width_in = 7.0
figure_height_in = 8.7

left_panel_width_ratio = 1.0
right_panel_width_ratio = 0.72

left_panel_hspace = 0.05

colorbar_pad = 0.025
colorbar_height_frac = 0.45

colorbar_label = (
    "Contribution to KL divergence from 277 K"
)


# --------------------------
# Shared typography
# --------------------------

axis_label_fontsize = 9
tick_labelsize = 7
legend_fontsize = 6
panel_label_fontsize = 13


# ============================================================
# FORMATTING HELPERS
# ============================================================

def format_axes(
    ax,
    minor_ticks_x=2,
    minor_ticks_y=2,
    label_fontsize=10,
    tick_labelsize=8,
    title_fontsize=10,
    stroke_in=0.01,
):
    line_width = stroke_in * 72.0

    ax.minorticks_on()

    if ax.get_xscale() == "log":
        ax.xaxis.set_minor_locator(
            ticker.LogLocator(
                subs="auto"
            )
        )
    else:
        ax.xaxis.set_minor_locator(
            AutoMinorLocator(
                minor_ticks_x
            )
        )

    if ax.get_yscale() == "log":
        ax.yaxis.set_minor_locator(
            ticker.LogLocator(
                subs="auto"
            )
        )
    else:
        ax.yaxis.set_minor_locator(
            AutoMinorLocator(
                minor_ticks_y
            )
        )

    for spine in ax.spines.values():
        spine.set_linewidth(
            1.5
        )
        spine.set_color(
            "black"
        )

    ax.tick_params(
        which="major",
        width=line_width,
        length=4.5,
        direction="in",
        top=True,
        bottom=True,
        left=True,
        right=True,
    )

    ax.tick_params(
        which="minor",
        width=line_width * 0.75,
        length=2.25,
        direction="in",
        top=True,
        bottom=True,
        left=True,
        right=True,
    )

    ax.xaxis.label.set_size(
        label_fontsize
    )

    ax.yaxis.label.set_size(
        label_fontsize
    )

    for tick_label in (
        ax.get_xticklabels()
        + ax.get_yticklabels()
    ):
        tick_label.set_fontsize(
            tick_labelsize
        )

    ax.title.set_fontsize(
        title_fontsize
    )


def canonical_label(
    value,
):
    return str(
        value
    ).strip().lower()


def pretty_label(
    label,
    pretty_map,
):
    return str(
        pretty_map.get(
            str(label),
            str(label),
        )
    )


def reorder_cooling_columns(
    matrix_df,
    desired_order,
):
    desired_canon = [
        canonical_label(
            value
        )
        for value in desired_order
    ]

    column_map = {
        canonical_label(
            column
        ): column
        for column in matrix_df.columns.tolist()
    }

    front = [
        column_map[value]
        for value in desired_canon
        if value in column_map
    ]

    rest = [
        column
        for column in matrix_df.columns.tolist()
        if column not in front
    ]

    return matrix_df.reindex(
        columns=front + rest
    )


# ============================================================
# INPUT LOADERS
# ============================================================

def load_event_time_csv(
    path,
):
    if not path.is_file():
        raise FileNotFoundError(
            f"Event-time CSV not found: {path}"
        )

    dataframe = pd.read_csv(
        path
    ).copy()

    if "state" not in dataframe.columns:
        raise ValueError(
            "Event-time CSV must contain a "
            "'state' column."
        )

    median_columns = [
        column
        for column in dataframe.columns
        if column.startswith(
            "event_time_median_"
        )
    ]

    low_columns = [
        column
        for column in dataframe.columns
        if column.startswith(
            "event_time_q025_"
        )
    ]

    high_columns = [
        column
        for column in dataframe.columns
        if column.startswith(
            "event_time_q975_"
        )
    ]

    if (
        len(median_columns) != 1
        or len(low_columns) != 1
        or len(high_columns) != 1
    ):
        raise ValueError(
            "Expected exactly one median, q025 and q975 "
            "event-time column.\n"
            f"Median: {median_columns}\n"
            f"q025: {low_columns}\n"
            f"q975: {high_columns}"
        )

    median_column = median_columns[0]
    low_column = low_columns[0]
    high_column = high_columns[0]

    suffix = median_column.removeprefix(
        "event_time_median_"
    )

    if "_" not in suffix:
        raise ValueError(
            f"Could not parse style and unit from "
            f"{median_column!r}."
        )

    event_style, time_unit = suffix.rsplit(
        "_",
        1,
    )

    dataframe["state"] = dataframe[
        "state"
    ].astype(
        int
    )

    dataframe = (
        dataframe.sort_values(
            "state"
        )
        .reset_index(
            drop=True
        )
    )

    return (
        dataframe,
        median_column,
        low_column,
        high_column,
        event_style,
        time_unit,
    )


def load_cooling_csv(
    path,
):
    if not path.is_file():
        raise FileNotFoundError(
            f"Cooling CSV not found: {path}"
        )

    dataframe = pd.read_csv(
        path
    ).copy()

    required_columns = {
        "state",
        "lab",
        "mean",
        "ci_lower_95",
        "ci_upper_95",
    }

    missing_columns = (
        required_columns
        - set(
            dataframe.columns
        )
    )

    if missing_columns:
        raise ValueError(
            "Cooling CSV missing required columns: "
            f"{sorted(missing_columns)}"
        )

    dataframe["state"] = dataframe[
        "state"
    ].astype(
        int
    )

    dataframe["lab"] = (
        dataframe["lab"]
        .astype(
            str
        )
        .str.strip()
    )

    return dataframe


# ============================================================
# HEATMAP CALCULATION HELPERS
# ============================================================

def baseline_series(
    dataframe,
    baseline_label_value,
    state_count,
):
    mask = (
        dataframe["lab"]
        .str.lower()
        .eq(
            str(
                baseline_label_value
            ).lower()
        )
    )

    baseline = (
        dataframe.loc[
            mask,
            [
                "state",
                "mean",
            ],
        ]
        .set_index(
            "state"
        )[
            "mean"
        ]
        .reindex(
            range(
                state_count
            )
        )
    )

    if baseline.isna().any():
        missing_states = (
            baseline[
                baseline.isna()
            ]
            .index
            .tolist()
        )

        raise ValueError(
            f"Baseline {baseline_label_value!r} "
            f"is missing states: {missing_states}"
        )

    return baseline


def change_tvd_contrib(
    end,
    base,
    end_low,
    end_high,
    eps,
):
    center = (
        0.5
        * np.abs(
            end - base
        )
    )

    low = (
        0.5
        * np.abs(
            end_low - base
        )
    )

    high = (
        0.5
        * np.abs(
            end_high - base
        )
    )

    return (
        center,
        low,
        high,
    )


def change_kl_qp_contrib(
    end,
    base,
    end_low,
    end_high,
    eps,
):
    q = np.maximum(
        end,
        eps,
    )

    p = np.maximum(
        base,
        eps,
    )

    q_low = np.maximum(
        end_low,
        eps,
    )

    q_high = np.maximum(
        end_high,
        eps,
    )

    center = (
        q
        * np.log(
            q / p
        )
    )

    low = (
        q_low
        * np.log(
            q_low / p
        )
    )

    high = (
        q_high
        * np.log(
            q_high / p
        )
    )

    return (
        center,
        low,
        high,
    )


def change_js_contrib(
    end,
    base,
    end_low,
    end_high,
    eps,
):
    p = np.maximum(
        base,
        eps,
    )

    q = np.maximum(
        end,
        eps,
    )

    q_low = np.maximum(
        end_low,
        eps,
    )

    q_high = np.maximum(
        end_high,
        eps,
    )

    midpoint = np.maximum(
        0.5
        * (
            p + q
        ),
        eps,
    )

    midpoint_low = np.maximum(
        0.5
        * (
            p + q_low
        ),
        eps,
    )

    midpoint_high = np.maximum(
        0.5
        * (
            p + q_high
        ),
        eps,
    )

    center = (
        0.5
        * p
        * np.log(
            p / midpoint
        )
        + 0.5
        * q
        * np.log(
            q / midpoint
        )
    )

    low = (
        0.5
        * p
        * np.log(
            p / midpoint_low
        )
        + 0.5
        * q_low
        * np.log(
            q_low / midpoint_low
        )
    )

    high = (
        0.5
        * p
        * np.log(
            p / midpoint_high
        )
        + 0.5
        * q_high
        * np.log(
            q_high / midpoint_high
        )
    )

    return (
        center,
        low,
        high,
    )


def change_relative_change(
    end,
    base,
    end_low,
    end_high,
    eps,
):
    base_values = np.asarray(
        base,
        dtype=float,
    )

    denominator = np.maximum(
        base_values,
        float(
            eps
        ),
    )

    center = (
        np.asarray(
            end,
            dtype=float,
        )
        - base_values
    ) / denominator

    low = (
        np.asarray(
            end_low,
            dtype=float,
        )
        - base_values
    ) / denominator

    high = (
        np.asarray(
            end_high,
            dtype=float,
        )
        - base_values
    ) / denominator

    return (
        center,
        low,
        high,
    )


change_metrics = {
    "tvd_contrib": change_tvd_contrib,
    "js_contrib": change_js_contrib,
    "kl_qp_contrib": change_kl_qp_contrib,
    "rel_change": change_relative_change,
    "relative_change": change_relative_change,
    "relative_median_change": change_relative_change,
}


def build_change_matrices(
    dataframe,
    baseline,
    exclude_labels,
    baseline_label_value,
    change_metric_value,
    eps,
    cooling_order_value,
):
    excluded = {
        canonical_label(
            value
        )
        for value in exclude_labels
    }

    excluded.add(
        canonical_label(
            baseline_label_value
        )
    )

    dataframe_use = dataframe.loc[
        ~dataframe["lab"]
        .str.lower()
        .isin(
            excluded
        )
    ].copy()

    dataframe_use = dataframe_use.merge(
        baseline.rename(
            "baseline_mean"
        ).reset_index(),
        on="state",
        how="left",
    )

    metric_key = canonical_label(
        change_metric_value
    )

    if metric_key not in change_metrics:
        raise ValueError(
            f"Unknown change_metric "
            f"{change_metric_value!r}."
        )

    metric_function = change_metrics[
        metric_key
    ]

    end = dataframe_use[
        "mean"
    ].to_numpy(
        dtype=float
    )

    end_low = dataframe_use[
        "ci_lower_95"
    ].to_numpy(
        dtype=float
    )

    end_high = dataframe_use[
        "ci_upper_95"
    ].to_numpy(
        dtype=float
    )

    baseline_values = dataframe_use[
        "baseline_mean"
    ].to_numpy(
        dtype=float
    )

    (
        center,
        low,
        high,
    ) = metric_function(
        end,
        baseline_values,
        end_low,
        end_high,
        float(
            eps
        ),
    )

    dataframe_use["center"] = center
    dataframe_use["low"] = low
    dataframe_use["high"] = high

    state_count = int(
        baseline.shape[0]
    )

    center_dataframe = (
        dataframe_use.pivot(
            index="state",
            columns="lab",
            values="center",
        )
        .reindex(
            index=range(
                state_count
            )
        )
    )

    low_dataframe = (
        dataframe_use.pivot(
            index="state",
            columns="lab",
            values="low",
        )
        .reindex(
            index=range(
                state_count
            )
        )
    )

    high_dataframe = (
        dataframe_use.pivot(
            index="state",
            columns="lab",
            values="high",
        )
        .reindex(
            index=range(
                state_count
            )
        )
    )

    return (
        reorder_cooling_columns(
            center_dataframe,
            cooling_order_value,
        ),
        reorder_cooling_columns(
            low_dataframe,
            cooling_order_value,
        ),
        reorder_cooling_columns(
            high_dataframe,
            cooling_order_value,
        ),
    )


def compute_vmin_vmax(
    values,
    scale_mode_value,
    vmax_percentile_value,
):
    scale_mode_value = canonical_label(
        scale_mode_value
    )

    if scale_mode_value == "nonnegative":
        vmax_local = float(
            np.nanpercentile(
                values,
                float(
                    vmax_percentile_value
                ),
            )
        )

        if (
            not np.isfinite(
                vmax_local
            )
            or vmax_local <= 0.0
        ):
            vmax_local = 1.0

        return (
            0.0,
            vmax_local,
        )

    if scale_mode_value == "minmax":
        vmin_local = float(
            np.nanmin(
                values
            )
        )

        vmax_local = float(
            np.nanmax(
                values
            )
        )

        if vmin_local == vmax_local:
            vmax_local = (
                vmin_local + 1.0
            )

        return (
            vmin_local,
            vmax_local,
        )

    if scale_mode_value == "symmetric":
        absolute_limit = float(
            np.nanpercentile(
                np.abs(
                    values
                ),
                float(
                    vmax_percentile_value
                ),
            )
        )

        if (
            not np.isfinite(
                absolute_limit
            )
            or absolute_limit <= 0.0
        ):
            absolute_limit = 1.0

        return (
            -absolute_limit,
            absolute_limit,
        )

    raise ValueError(
        "scale_mode must be one of: "
        "'nonnegative', 'symmetric', 'minmax'"
    )


def text_color_from_cmap(
    value,
    cmap_name,
    norm,
):
    cmap_object = plt.get_cmap(
        str(
            cmap_name
        )
    )

    red, green, blue, _ = cmap_object(
        norm(
            float(
                value
            )
        )
    )

    luminance = (
        0.2126 * red
        + 0.7152 * green
        + 0.0722 * blue
    )

    return (
        "white"
        if luminance < 0.5
        else "black"
    )


# ============================================================
# HISTOGRAM HELPERS
# ============================================================

def extract_series_from_csv(
    dataframe,
    label,
    state_count,
):
    mask = (
        dataframe["lab"]
        .astype(
            str
        )
        .str.strip()
        .str.lower()
        .eq(
            str(
                label
            )
            .strip()
            .lower()
        )
    )

    subset = (
        dataframe.loc[
            mask
        ]
        .sort_values(
            "state"
        )
    )

    if subset.empty:
        raise ValueError(
            f"No rows found for lab={label!r}."
        )

    states = subset[
        "state"
    ].to_numpy(
        dtype=int
    )

    expected_states = np.arange(
        state_count
    )

    if not np.array_equal(
        states,
        expected_states,
    ):
        raise ValueError(
            f"Rows for lab={label!r} do not cover "
            f"states 0..{state_count - 1}."
        )

    mean = subset[
        "mean"
    ].to_numpy(
        dtype=float
    )

    lower = subset[
        "ci_lower_95"
    ].to_numpy(
        dtype=float
    )

    upper = subset[
        "ci_upper_95"
    ].to_numpy(
        dtype=float
    )

    if (
        np.all(
            np.isnan(
                lower
            )
        )
        and np.all(
            np.isnan(
                upper
            )
        )
    ):
        confidence_interval = None
    else:
        confidence_interval = np.vstack(
            (
                lower,
                upper,
            )
        )

    return (
        mean,
        confidence_interval,
    )


def yerr_from_mean_ci(
    mean,
    confidence_interval,
):
    if confidence_interval is None:
        return None

    lower_error = np.maximum(
        mean
        - confidence_interval[
            0
        ],
        0.0,
    )

    upper_error = np.maximum(
        confidence_interval[
            1
        ]
        - mean,
        0.0,
    )

    return np.vstack(
        (
            lower_error,
            upper_error,
        )
    )


def compute_ymax_for_states(
    states,
    pi_mean,
    pi_ci,
    cooled_mean,
    cooled_ci,
    cooling_labels,
    pi_230k,
    pi_230k_ci,
    pad_fraction,
):
    ymax = 0.0

    if pi_ci is not None:
        ymax = max(
            ymax,
            float(
                np.max(
                    pi_ci[
                        1,
                        states,
                    ]
                )
            ),
        )
    else:
        ymax = max(
            ymax,
            float(
                np.max(
                    pi_mean[
                        states
                    ]
                )
            ),
        )

    for label in cooling_labels:
        confidence_interval = cooled_ci[
            label
        ]

        if confidence_interval is not None:
            values = confidence_interval[
                1,
                states,
            ]
        else:
            values = cooled_mean[
                label
            ][
                states
            ]

        ymax = max(
            ymax,
            float(
                np.max(
                    values
                )
            ),
        )

    if pi_230k is not None:
        if pi_230k_ci is not None:
            values = pi_230k_ci[
                1,
                states,
            ]
        else:
            values = pi_230k[
                states
            ]

        ymax = max(
            ymax,
            float(
                np.max(
                    values
                )
            ),
        )

    if (
        not np.isfinite(
            ymax
        )
        or ymax <= 0.0
    ):
        ymax = 1.0

    return (
        ymax
        * (
            1.0
            + pad_fraction
        )
    )


def plot_bars_on_ax(
    ax,
    positions,
    states,
    pi_mean,
    pi_ci,
    cooled_mean,
    cooled_ci,
    cooling_labels,
    include_230k,
    pi_230k,
    pi_230k_ci,
    colours,
    offsets,
    bar_width,
):
    equilibrium_mean = pi_mean[
        states
    ]

    if pi_ci is not None:
        equilibrium_ci = pi_ci[
            :,
            states,
        ]
    else:
        equilibrium_ci = None

    equilibrium_yerr = yerr_from_mean_ci(
        equilibrium_mean,
        equilibrium_ci,
    )

    ax.bar(
        positions
        + offsets[
            0
        ],
        equilibrium_mean,
        bar_width,
        color=colours[
            0
        ],
        alpha=1.0,
        yerr=equilibrium_yerr,
        capsize=(
            err_capsize
            if equilibrium_yerr is not None
            else 0
        ),
        error_kw=(
            err_kw
            if equilibrium_yerr is not None
            else None
        ),
    )

    for series_index, label in enumerate(
        cooling_labels,
        start=1,
    ):
        mean = cooled_mean[
            label
        ][
            states
        ]

        if cooled_ci[label] is not None:
            confidence_interval = cooled_ci[
                label
            ][
                :,
                states,
            ]
        else:
            confidence_interval = None

        yerr = yerr_from_mean_ci(
            mean,
            confidence_interval,
        )

        ax.bar(
            positions
            + offsets[
                series_index
            ],
            mean,
            bar_width,
            color=colours[
                series_index
            ],
            alpha=1.0,
            yerr=yerr,
            capsize=(
                err_capsize
                if yerr is not None
                else 0
            ),
            error_kw=(
                err_kw
                if yerr is not None
                else None
            ),
        )

    if include_230k:
        series_index = (
            len(
                colours
            )
            - 1
        )

        mean = pi_230k[
            states
        ]

        if pi_230k_ci is not None:
            confidence_interval = pi_230k_ci[
                :,
                states,
            ]
        else:
            confidence_interval = None

        yerr = yerr_from_mean_ci(
            mean,
            confidence_interval,
        )

        ax.bar(
            positions
            + offsets[
                series_index
            ],
            mean,
            bar_width,
            color=colours[
                series_index
            ],
            alpha=1.0,
            yerr=yerr,
            capsize=(
                err_capsize
                if yerr is not None
                else 0
            ),
            error_kw=(
                err_kw
                if yerr is not None
                else None
            ),
        )


# ============================================================
# LOAD INPUTS
# ============================================================

(
    event_dataframe,
    event_median_column,
    event_low_column,
    event_high_column,
    event_style,
    time_unit,
) = load_event_time_csv(
    event_time_csv
)

cooling_dataframe = load_cooling_csv(
    cooling_csv
)

state_count = (
    int(
        event_dataframe[
            "state"
        ].max()
    )
    + 1
)

if state_count != k_choice:
    raise ValueError(
        f"event_time_csv contains {state_count} states, "
        f"but k_choice={k_choice}."
    )

if canonical_label(
    event_style
) != "da":
    raise ValueError(
        f"Expected DA event times, "
        f"found style={event_style!r}."
    )

if canonical_label(
    time_unit
) != "ns":
    raise ValueError(
        f"Expected event-time values in ns, "
        f"found unit={time_unit!r}."
    )

expected_states = np.arange(
    k_choice
)

event_states = event_dataframe[
    "state"
].to_numpy(
    dtype=int
)

if not np.array_equal(
    event_states,
    expected_states,
):
    raise ValueError(
        "Event-time CSV states are not contiguous "
        f"from 0 to {k_choice - 1}."
    )

cooling_states = np.sort(
    cooling_dataframe[
        "state"
    ].unique()
)

if not np.array_equal(
    cooling_states,
    expected_states,
):
    raise ValueError(
        "Cooling CSV states are not contiguous "
        f"from 0 to {k_choice - 1}."
    )


# ============================================================
# PANEL A DATA
# ============================================================

pi_mean, pi_ci = extract_series_from_csv(
    cooling_dataframe,
    "pi_277K",
    k_choice,
)

cooled_mean = {}
cooled_ci = {}

for label in cooling_order:
    (
        cooled_mean[
            label
        ],
        cooled_ci[
            label
        ],
    ) = extract_series_from_csv(
        cooling_dataframe,
        label,
        k_choice,
    )

include_230k = np.any(
    cooling_dataframe[
        "lab"
    ]
    .str.strip()
    .str.lower()
    .eq(
        "pi_230k"
    )
)

if include_230k:
    (
        pi_230k,
        pi_230k_ci,
    ) = extract_series_from_csv(
        cooling_dataframe,
        "pi_230K",
        k_choice,
    )
else:
    pi_230k = None
    pi_230k_ci = None

state_order = np.argsort(
    pi_mean
)[
    ::-1
]

series_keys = (
    [
        "277K",
    ]
    + cooling_order
    + (
        [
            "230K",
        ]
        if include_230k
        else []
    )
)

series_labels = (
    [
        "Equilibrium 277 K",
    ]
    + [
        cooling_pretty_labels.get(
            label,
            label,
        )
        for label in cooling_order
    ]
    + (
        [
            "Equilibrium 230 K",
        ]
        if include_230k
        else []
    )
)

series_count = len(
    series_keys
)

bar_width = (
    0.8
    / series_count
)

offsets = (
    np.arange(
        series_count
    )
    - (
        series_count - 1
    )
    / 2.0
) * bar_width

colours = [
    vit_colors.get(
        key,
        "0.5",
    )
    for key in series_keys
]

legend_handles = [
    Patch(
        facecolor=colours[
            index
        ],
        edgecolor="none",
        label=series_labels[
            index
        ],
    )
    for index in range(
        series_count
    )
]

states_per_row = int(
    math.ceil(
        k_choice
        / full_nrows
    )
)

row_specs = []

for row_index in range(
    full_nrows
):
    start_rank = (
        row_index
        * states_per_row
    )

    end_rank = min(
        (
            row_index + 1
        )
        * states_per_row,
        k_choice,
    )

    if start_rank >= k_choice:
        break

    row_ranks = np.arange(
        start_rank,
        end_rank,
    )

    row_states = state_order[
        row_ranks
    ]

    y_max_row = compute_ymax_for_states(
        states=row_states,
        pi_mean=pi_mean,
        pi_ci=pi_ci,
        cooled_mean=cooled_mean,
        cooled_ci=cooled_ci,
        cooling_labels=cooling_order,
        pi_230k=pi_230k,
        pi_230k_ci=pi_230k_ci,
        pad_fraction=full_ymax_pad_frac,
    )

    row_specs.append(
        (
            start_rank,
            end_rank,
            row_ranks,
            row_states,
            y_max_row,
        )
    )

active_row_count = len(
    row_specs
)


# ============================================================
# PANEL B DATA
# ============================================================

baseline = baseline_series(
    cooling_dataframe,
    baseline_label_value=baseline_label,
    state_count=k_choice,
)

(
    state_matrix,
    state_lower_matrix,
    state_upper_matrix,
) = build_change_matrices(
    cooling_dataframe,
    baseline,
    exclude_labels=exclude_labs,
    baseline_label_value=baseline_label,
    change_metric_value=change_metric,
    eps=1.0e-12,
    cooling_order_value=cooling_order,
)

event_median = (
    event_dataframe
    .set_index(
        "state"
    )[
        event_median_column
    ]
    .reindex(
        range(
            k_choice
        )
    )
)

if event_median.isna().any():
    missing_states = (
        event_median[
            event_median.isna()
        ]
        .index
        .tolist()
    )

    raise ValueError(
        "Event-time CSV is missing states: "
        f"{missing_states}"
    )

event_median_scaled = (
    event_median
    / float(
        event_time_divisor
    )
)

heatmap_order = np.argsort(
    event_median_scaled.to_numpy(
        dtype=float
    )
).astype(
    int
)

heatmap_matrix = state_matrix.reindex(
    index=heatmap_order
)

heatmap_lower_matrix = state_lower_matrix.reindex(
    index=heatmap_order
)

heatmap_upper_matrix = state_upper_matrix.reindex(
    index=heatmap_order
)

heatmap_xlabels = [
    pretty_label(
        label,
        cooling_pretty_labels,
    )
    for label in heatmap_matrix.columns.tolist()
]

state_number_width = max(
    2,
    len(
        str(
            k_choice - 1
        )
    ),
)

heatmap_ylabels = [
    (
        f"state {state_index:0{state_number_width}d} | "
        f"{event_median_scaled.iloc[state_index]:.{event_time_label_decimals}f} ns"
    )
    for state_index in heatmap_order
]

heatmap_values = heatmap_matrix.to_numpy(
    dtype=float
)

(
    heatmap_vmin,
    heatmap_vmax,
) = compute_vmin_vmax(
    heatmap_values,
    scale_mode,
    vmax_percentile,
)

heatmap_norm = Normalize(
    vmin=heatmap_vmin,
    vmax=heatmap_vmax,
    clip=True,
)


# ============================================================
# BUILD FIGURE
# ============================================================

fig = plt.figure(
    figsize=(
        figure_width_in,
        figure_height_in,
    ),
    constrained_layout=True,
)

outer_grid = fig.add_gridspec(
    nrows=1,
    ncols=2,
    width_ratios=[
        left_panel_width_ratio,
        right_panel_width_ratio,
    ],
    wspace=0.0,
)

left_grid = outer_grid[
    0,
    0,
].subgridspec(
    nrows=full_nrows + 1,
    ncols=1,
    hspace=left_panel_hspace,
    height_ratios=(
        [
            1.0,
        ]
        * full_nrows
        + [
            0.18,
        ]
    ),
)

left_axes = [
    fig.add_subplot(
        left_grid[
            row_index,
            0,
        ]
    )
    for row_index in range(
        full_nrows
    )
]

legend_ax = fig.add_subplot(
    left_grid[
        full_nrows,
        0,
    ]
)

right_ax = fig.add_subplot(
    outer_grid[
        0,
        1,
    ]
)


# ============================================================
# PANEL A: MULTI-ROW BAR CHARTS
# ============================================================

for row_index in range(
    full_nrows
):
    axis = left_axes[
        row_index
    ]

    if row_index >= active_row_count:
        axis.set_axis_off()
        continue

    (
        start_rank,
        end_rank,
        row_ranks,
        row_states,
        y_max_row,
    ) = row_specs[
        row_index
    ]

    positions = row_ranks.astype(
        float
    )

    plot_bars_on_ax(
        ax=axis,
        positions=positions,
        states=row_states,
        pi_mean=pi_mean,
        pi_ci=pi_ci,
        cooled_mean=cooled_mean,
        cooled_ci=cooled_ci,
        cooling_labels=cooling_order,
        include_230k=include_230k,
        pi_230k=pi_230k,
        pi_230k_ci=pi_230k_ci,
        colours=colours,
        offsets=offsets,
        bar_width=bar_width,
    )

    axis.set_xlim(
        start_rank - 0.5,
        end_rank - 0.5,
    )

    axis.set_ylim(
        0.0,
        y_max_row,
    )

    axis.set_xticks(
        row_ranks
    )

    axis.set_xticklabels(
        row_states,
        rotation=45,
        ha="right",
    )

    axis.set_ylabel(
        "Probability",
        fontsize=axis_label_fontsize,
    )

    if (
        row_index
        == active_row_count - 1
    ):
        axis.set_xlabel(
            "State index",
            fontsize=axis_label_fontsize,
        )

    format_axes(
        axis,
        label_fontsize=axis_label_fontsize,
        tick_labelsize=tick_labelsize,
        title_fontsize=axis_label_fontsize,
    )


if (
    legend_enabled
    and active_row_count > 0
):
    legend_ax.set_axis_off()

    legend_ax.legend(
        handles=legend_handles,
        loc="center",
        ncol=legend_ncol,
        frameon=False,
        fontsize=legend_fontsize,
        handlelength=legend_handlelength,
        handletextpad=legend_handletextpad,
        columnspacing=legend_columnspacing,
        borderpad=legend_borderpad,
    )
else:
    legend_ax.set_axis_off()


# ============================================================
# PANEL B: HEATMAP
# ============================================================

image = right_ax.imshow(
    heatmap_values,
    aspect="auto",
    interpolation="nearest",
    cmap=cmap,
    norm=heatmap_norm,
)

right_ax.set_xticks(
    np.arange(
        heatmap_matrix.shape[
            1
        ]
    )
)

right_ax.set_xticklabels(
    heatmap_xlabels,
    rotation=45,
    ha="right",
    fontsize=tick_labelsize,
)

right_ax.set_yticks(
    np.arange(
        heatmap_matrix.shape[
            0
        ]
    )
)

right_ax.set_yticklabels(
    heatmap_ylabels,
    fontsize=tick_labelsize,
)

right_ax.set_xlabel(
    "Cooling rate",
    fontsize=axis_label_fontsize,
)

right_ax.set_ylabel(
    "State | event time",
    fontsize=axis_label_fontsize,
    labelpad=0,
)

format_axes(
    right_ax,
    label_fontsize=axis_label_fontsize,
    tick_labelsize=tick_labelsize,
    title_fontsize=axis_label_fontsize,
)

right_ax.tick_params(
    axis="both",
    which="both",
    length=0,
)


if annotate:
    number_format = (
        f"{{:.{int(annotate_decimals)}f}}"
    )

    for row_index in range(
        heatmap_matrix.shape[
            0
        ]
    ):
        for column_index in range(
            heatmap_matrix.shape[
                1
            ]
        ):
            value = float(
                heatmap_matrix.iloc[
                    row_index,
                    column_index,
                ]
            )

            lower_value = float(
                heatmap_lower_matrix.iloc[
                    row_index,
                    column_index,
                ]
            )

            upper_value = float(
                heatmap_upper_matrix.iloc[
                    row_index,
                    column_index,
                ]
            )

            if not np.isfinite(
                value
            ):
                continue

            label_colour = text_color_from_cmap(
                value,
                cmap_name=cmap,
                norm=heatmap_norm,
            )

            if (
                annotate_with_ci
                and np.isfinite(
                    lower_value
                )
                and np.isfinite(
                    upper_value
                )
            ):
                annotation_text = (
                    f"{number_format.format(value)}\n"
                    f"["
                    f"{number_format.format(lower_value)},"
                    f"{number_format.format(upper_value)}"
                    f"]"
                )
            else:
                annotation_text = number_format.format(
                    value
                )

            right_ax.text(
                column_index,
                row_index,
                annotation_text,
                ha="center",
                va="center",
                fontsize=float(
                    annotate_fontsize
                ),
                color=label_colour,
            )


# ============================================================
# COLORBAR
# ============================================================

colorbar = fig.colorbar(
    image,
    ax=right_ax,
    fraction=0.046,
    pad=colorbar_pad,
    shrink=colorbar_height_frac,
    aspect=18,
)

colorbar.ax.tick_params(
    labelsize=7
)

colorbar.set_label(
    colorbar_label,
    fontsize=axis_label_fontsize,
    labelpad=1,
)


# ============================================================
# PANEL LABELS
# ============================================================

label_settings = {
    "xycoords": "axes fraction",
    "fontsize": panel_label_fontsize,
    "va": "bottom",
    "ha": "left",
    "annotation_clip": False,
}

left_axes[
    0
].annotate(
    "A",
    xy=(
        -0.12,
        1.02,
    ),
    **label_settings,
)

right_ax.annotate(
    "B",
    xy=(
        -0.22,
        1.02,
    ),
    **label_settings,
)


# ============================================================
# SAVE AND DISPLAY
# ============================================================

fig.savefig(
    figure_path,
    format="pdf",
)

print(
    f"State count: {k_choice}"
)

print(
    f"Cooling CSV: {cooling_csv}"
)

print(
    f"Event-time CSV: {event_time_csv}"
)

print(
    f"Saved supplementary figure to: "
    f"{figure_path}"
)

plt.show()